# Triadic Neurosymbolic Engine — Reproducibility Demo

This notebook demonstrates the core features of the Triadic Neurosymbolic Engine:

1. **Prime Encoding** — Project words into composite prime integers
2. **4 Projection Modes** — Random, PCA, Consensus, Contrastive
3. **Algebraic Operations** — Subsumption, Composition, Gap Analysis
4. **Anomaly Detection** — Verify multiplicative invariants in tabular data
5. **Model Auditing** — Compare two embedding models structurally

In [ ]:
# Install (run once)
# !pip install -e ..

In [ ]:
import sys, math
sys.path.insert(0, '../src')

from neurosym import (
    ContinuousEncoder, DiscreteMapper, DiscreteValidator,
    ScalableGraphBuilder, AnomalyDetector, RelationalRule
)
import pandas as pd
import numpy as np

## 1. Prime Encoding

Each word is embedded into a 384-dim vector, then projected through LSH hyperplanes.
Each hyperplane maps to a unique prime. The word's integer = product of its active primes.

In [ ]:
encoder = ContinuousEncoder("all-MiniLM-L6-v2")
concepts = ["King", "Queen", "Man", "Woman", "Dog", "Cat", "Animal", "Car", "Vehicle"]

embeddings = encoder.encode(concepts)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
mapper = DiscreteMapper(n_bits=8, projection="pca")
prime_map = mapper.fit_transform(concepts, embeddings)

for word, integer in prime_map.items():
    from sympy import factorint
    factors = factorint(integer)
    factor_str = ' × '.join(f'{p}' for p in sorted(factors.keys()))
    print(f"{word:>8} = {integer:>8}  ({factor_str})")

## 2. Projection Mode Comparison

The engine supports 4 projection modes. Let's compare them on subsumption detection:

In [ ]:
hypernym_pairs = [
    ("Animal", "Dog"), ("Animal", "Cat"),
    ("Vehicle", "Car")
]

modes = {
    "random":      DiscreteMapper(n_bits=8, seed=42, projection="random"),
    "pca":         DiscreteMapper(n_bits=8, projection="pca"),
    "consensus":   DiscreteMapper(n_bits=8, projection="consensus", consensus_threshold=0.7),
    "contrastive": DiscreteMapper(n_bits=8, projection="contrastive",
                                  hypernym_pairs=hypernym_pairs),
}

print(f"{'Mode':>14} | {'Animal⊇Dog':>12} | {'Animal⊇Cat':>12} | {'Vehicle⊇Car':>12}")
print("-" * 60)

for name, m in modes.items():
    pm = m.fit_transform(concepts, embeddings)
    results = []
    for broader, narrower in hypernym_pairs:
        ok = pm[broader] % pm[narrower] == 0
        results.append('✓' if ok else '✗')
    print(f"{name:>14} | {results[0]:>12} | {results[1]:>12} | {results[2]:>12}")

## 3. Algebraic Operations

Three operations **impossible** with cosine similarity:

In [ ]:
v = DiscreteValidator()

king, queen = prime_map["King"], prime_map["Queen"]
man, woman = prime_map["Man"], prime_map["Woman"]

# --- Subsumption ---
print("=== Subsumption ===")
print(f"King ⊇ Queen? {v.subsumes(king, queen)}")
print(f"Queen ⊇ King? {v.subsumes(queen, king)}")

# --- Composition ---
print("\n=== Composition ===")
royal_male = v.compose(king, man)
print(f"compose(King, Man) = {royal_male}")
print(f"Contains King features? {royal_male % king == 0}")
print(f"Contains Man features?  {royal_male % man == 0}")

# --- Gap Analysis ---
print("\n=== Gap Analysis ===")
gap = v.explain_gap(king, queen)
print(f"King vs Queen:")
print(f"  Shared backbone (GCD):  {gap['shared']}")
print(f"  Only in King:           {gap['only_in_a']}")
print(f"  Only in Queen:          {gap['only_in_b']}")
print(f"  King contains Queen?    {gap['a_contains_b']}")
print(f"  Queen contains King?    {gap['b_contains_a']}")

## 4. Semantic Graph Building

The `ScalableGraphBuilder` uses an inverted prime index — O(N×F×B) instead of O(N²):

In [ ]:
graph = ScalableGraphBuilder()
graph.build_index(prime_map)

edges = graph.find_edges(prime_map, min_shared=2)
print(f"Edges with ≥2 shared factors: {len(edges)}\n")

for a, b, weight, shared in edges[:10]:
    print(f"  {a:>8} ↔ {b:<8}  weight={weight}  shared_primes={shared}")

print(f"\nIndex stats: {graph.get_stats()}")

## 5. Anomaly Detection (Tabular Data)

The engine can verify multiplicative invariants in real data — no embeddings needed:

In [ ]:
# Sample invoice data with an intentional error in row 2
df = pd.DataFrame({
    "item":       ["Widget A", "Widget B", "Widget C", "Widget D"],
    "qty":        [10, 5, 3, 7],
    "unit_price": [25.0, 40.0, 100.0, 15.0],
    "tax_rate":   [1.16, 1.16, 1.16, 1.16],
    "total":      [290.0, 232.0, 999.0, 121.80]  # Row 2 is WRONG (should be 348.0)
})

detector = AnomalyDetector()
detector.add_rule(RelationalRule(
    name="Invoice Total",
    factor_columns=["qty", "unit_price", "tax_rate"],
    result_column="total"
))

anomalies = detector.scan(df)
for a in anomalies:
    print(f"[{a.severity}] Row {a.row_index}: {a.explanation}")

## 6. Model Auditing

Compare how two different embedding models structure the same concepts:

In [ ]:
# Encode with two different models
encoder_a = ContinuousEncoder("all-MiniLM-L6-v2")
encoder_b = ContinuousEncoder("paraphrase-MiniLM-L3-v2")

test_words = ["Doctor", "Nurse", "Teacher", "Happy", "Sad"]

emb_a = encoder_a.encode(test_words)
emb_b = encoder_b.encode(test_words)

mapper_a = DiscreteMapper(n_bits=8, projection="pca")
mapper_b = DiscreteMapper(n_bits=8, projection="pca")

pm_a = mapper_a.fit_transform(test_words, emb_a)
pm_b = mapper_b.fit_transform(test_words, emb_b)

print(f"{'Concept':>10} | {'Model A':>10} | {'Model B':>10} | {'GCD':>8} | {'Consensus':>10}")
print("-" * 60)
for w in test_words:
    gcd = math.gcd(pm_a[w], pm_b[w])
    consensus = '✓' if pm_a[w] == pm_b[w] else '✗'
    print(f"{w:>10} | {pm_a[w]:>10} | {pm_b[w]:>10} | {gcd:>8} | {consensus:>10}")

---

**Paper:** See `paper/src/main.tex` for the full 9-experiment evaluation.  
**Dashboard:** Run `streamlit run app.py` for an interactive UI.  
**Citation:** See `README.md` for the BibTeX block.